# Predicting Airline Passenger Satisfaction (Kaggle Playground Series s6e10)
### Top Solution Pipeline: 10-Fold Stratified CV, 73 Sovereign Features, max_bin=512 Histogram Trees & Multi-Champion Ensembling

This notebook implements the top-performing pipeline:
- **73 Sovereign Features**: 63 static features (route profiles by `Flight Distance`, delays, service interactions, frequency encodings, optional clean prior) + 10 zero-leakage in-fold Bayesian Target Encodings.
- **High-Resolution Histogram Binning**: `max_bin=512` on continuous route and survey metrics.
- **Diverse Ensemble Suite**:
  1. `XGBoost_Depth8_Wide` (`colsample_bytree=0.60`, `colsample_bynode=0.65`, `min_child_weight=30`, `reg_lambda=2.5`, `max_bin=512`)
  2. `XGBoost_Tuned_D9` (`max_depth=9`, `colsample_bytree=0.70`, `colsample_bynode=0.50`, `min_child_weight=55`, `reg_lambda=3.55`, `max_bin=512`)
  3. `XGBoost_Depth10_Anchor` (`max_depth=10`, `subsample=0.85`, `colsample_bytree=0.80`, `min_child_weight=20`, `reg_lambda=3.0`)
  4. `LightGBM_Deep` (`num_leaves=255`, `max_depth=10`, `min_child_samples=60`, `subsample=0.80`, `colsample_bytree=0.75`, `reg_lambda=4.0`)
- **Ensemble Blends**: Caruana ROC AUC Forward Selection, Pure XGBoost Trio (45/45/10), Pure Duo (50/50), 50/50 Consensus Champion, and Percentile Rank-Normalized predictions.

## 1. Environment & Reproducibility Setup

In [ ]:
import os, sys, glob, gc, time, math, warnings, random
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# Hardware profiling
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
# Set SMOKE = True for a rapid test run, False for complete production training
SMOKE = os.environ.get('SMOKE', '0') == '1'
N_FOLDS = 10 if not SMOKE else 3
N_EST = 3500 if GPU else (250 if SMOKE else 1600)
ES = 150 if GPU else (40 if SMOKE else 100)
CPU_THREADS = os.cpu_count() or 4

print(f'[*] Execution Environment Active:')
print(f' * Device: {DEVICE.upper()} (GPU: {GPU})')
if GPU:
    print(f' * GPU Model: {torch.cuda.get_device_name(0)}')
print(f' * Smoke Mode: {SMOKE}')
print(f' * Folds: {N_FOLDS} | Tree Budget: {N_EST} | Early Stopping: {ES}')

## 2. Dataset Discovery & Loading

In [ ]:
def find_path(patterns):
    for p in patterns:
        m = glob.glob(p)
        if m:
            return m[0]
    return None

train_path = find_path([
    'data/raw/train.csv',
    'data/train.csv',
    '/kaggle/input/competitions/playground-series-s6e10/train.csv',
    '/kaggle/input/playground-series-s6e10/train.csv',
    'train.csv'
])

test_path = find_path([
    'data/raw/test.csv',
    'data/test.csv',
    '/kaggle/input/competitions/playground-series-s6e10/test.csv',
    '/kaggle/input/playground-series-s6e10/test.csv',
    'test.csv'
])

sub_path = find_path([
    'data/raw/sample_submission.csv',
    'data/sample_submission.csv',
    '/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv',
    '/kaggle/input/playground-series-s6e10/sample_submission.csv',
    'sample_submission.csv'
])

orig_path = find_path([
    'data/raw/data.csv',
    'data/data.csv',
    '/kaggle/input/datasets/arseniyshutko/binary-aviation-satisfaction-129k/data.csv',
    '/kaggle/input/binary-aviation-satisfaction-129k/data.csv',
    '/kaggle/input/airline-passenger-satisfaction/train.csv',
    'Airline Satisfaction Data*/data.csv'
])

print('Resolved Paths:')
print(f' * Train: {train_path}')
print(f' * Test:  {test_path}')
print(f' * Sample Sub: {sub_path}')
print(f' * Orig (Optional External): {orig_path}')

assert train_path and test_path, 'Error: Train or Test dataset missing.'

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
orig_raw = pd.read_csv(orig_path) if orig_path else pd.DataFrame()

if SMOKE:
    print('[!] Smoke Active: Subsetting datasets for quick pipeline check...')
    train_raw = train_raw.head(8000).copy()
    test_raw = test_raw.head(2000).copy()
    if not orig_raw.empty:
        orig_raw = orig_raw.head(3000).copy()

TARGET = 'satisfaction'
FD = 'Flight Distance'

# Standardize Target Labels to Binary Integer
if train_raw[TARGET].dtype == object:
    train_raw[TARGET] = (train_raw[TARGET].astype(str).str.lower().str.strip() == 'satisfied').astype(np.int32)

if not orig_raw.empty and TARGET in orig_raw.columns:
    if orig_raw[TARGET].dtype == object:
        orig_raw[TARGET] = (orig_raw[TARGET].astype(str).str.lower().str.strip() == 'satisfied').astype(np.int32)
    y_orig = orig_raw[TARGET].values.astype(np.int32)
else:
    y_orig = np.array([], dtype=np.int32)

y_train = train_raw[TARGET].values.astype(np.int32)
print(f'Loaded: Train={train_raw.shape}, Test={test_raw.shape}, Target Mean={y_train.mean():.4f}')

## 3. Feature Engineering: 63 Static Features
Constructs core columns, delay dynamics, engagement, route profile aggregations on `Flight Distance`, and external priors (if available).

In [ ]:
NTR = len(train_raw)
NTE = len(test_raw)
full_df = pd.concat([train_raw.drop(columns=[TARGET], errors='ignore'), test_raw], ignore_index=True)

CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', FD, 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness'
]
CORE_FEATS = NUMS + RATINGS + CATS

# 1. Missing Value Imputation
full_df['Arrival Delay in Minutes'] = full_df['Arrival Delay in Minutes'].fillna(full_df['Departure Delay in Minutes']).fillna(0.0)

# 2. Delay Dynamics & Survey Engagement
full_df['Delay_Total'] = (full_df['Departure Delay in Minutes'] + full_df['Arrival Delay in Minutes']).astype(np.float32)
full_df['Delay_Diff'] = (full_df['Arrival Delay in Minutes'] - full_df['Departure Delay in Minutes']).astype(np.float32)
full_df['Delay_Ratio'] = ((full_df['Arrival Delay in Minutes'] + 1.0) / (full_df['Departure Delay in Minutes'] + 1.0)).astype(np.float32)
full_df['Delay_Per_Mile'] = (full_df['Delay_Total'] / (full_df[FD] + 1.0)).astype(np.float32)
full_df['Zero_Ratings_Count'] = (full_df[RATINGS] == 0).sum(axis=1).astype(np.float32)

# 3. Flight Context & Service Interactions
full_df['Digital_Service_Index'] = ((full_df['Inflight wifi service'] + 1.0) * (full_df['Online boarding'] + 1.0)).astype(np.float32)
full_df['Digital_Satisfaction_Gap'] = (full_df['Inflight wifi service'] - full_df['Online boarding']).astype(np.float32)
full_df['Physical_Service_Index'] = ((full_df['Seat comfort'] + full_df['Cleanliness'] + full_df['Food and drink']) / 3.0).astype(np.float32)
full_df['Cabin_Comfort_Cluster'] = ((full_df['Seat comfort'] + full_df['Leg room service'] + full_df['Cleanliness']) / 3.0).astype(np.float32)
class_map = {'Eco': 0, 'Eco Plus': 1, 'Business': 2}
class_numeric = full_df['Class'].map(class_map).fillna(0).astype(np.float32)
full_df['Service_Expectation_Gap'] = (full_df['Physical_Service_Index'] - (class_numeric * 1.5)).astype(np.float32)
full_df['Wifi_Boarding_Ratio'] = ((full_df['Inflight wifi service'] + 0.5) / (full_df['Online boarding'] + 0.5)).astype(np.float32)
loyalty_num = (full_df['Customer Type'] == 'Loyal Customer').astype(np.float32)
biz_num = (full_df['Type of Travel'] == 'Business travel').astype(np.float32)
full_df['Loyal_Business_Segment'] = (loyalty_num * 2.0 + biz_num).astype(np.float32)

# 4. Frequency Encodings
CNT_FEATS = pd.DataFrame(index=full_df.index)
for col in NUMS:
    CNT_FEATS[f'cnt_{col}'] = full_df[col].map(train_raw[col].value_counts()).fillna(0).astype(np.float32)

# 5. Route Profile Means Grouped by Flight Distance
coded_full = full_df[CORE_FEATS].copy()
for c in CATS:
    coded_full[c] = coded_full[c].astype('category').cat.codes
route_group = coded_full.groupby(FD)
ROUTE_PROFILES = pd.DataFrame(index=full_df.index)
for c in CORE_FEATS:
    if c != FD:
        ROUTE_PROFILES[f'route_mean_{c}'] = route_group[c].transform('mean').astype(np.float32)
ROUTE_PROFILES['route_size'] = route_group['Age'].transform('size').astype(np.float32)

# 6. External Priors (Gracefully handles absence of external dataset)
ORIG_PRIOR_FEATS = pd.DataFrame(index=full_df.index)
if not orig_raw.empty and len(y_orig) > 0:
    orig_stats_fd = pd.DataFrame({'fd': orig_raw[FD], 'y': y_orig}).groupby('fd')['y'].agg(['sum', 'count'])
    orig_global_prior = float(y_orig.mean())
    smoothed_orig_route = (orig_stats_fd['sum'] + orig_global_prior * 20.0) / (orig_stats_fd['count'] + 20.0)
    ORIG_PRIOR_FEATS['orig_route_satisfaction_prior'] = full_df[FD].map(smoothed_orig_route).fillna(orig_global_prior).astype(np.float32)

    orig_raw['Arrival Delay in Minutes'] = orig_raw['Arrival Delay in Minutes'].fillna(orig_raw['Departure Delay in Minutes']).fillna(0.0)
    orig_raw['Delay_Total'] = (orig_raw['Departure Delay in Minutes'] + orig_raw['Arrival Delay in Minutes']).astype(np.float32)
    orig_raw['Delay_Diff'] = (orig_raw['Arrival Delay in Minutes'] - orig_raw['Departure Delay in Minutes']).astype(np.float32)
    orig_raw['Delay_Ratio'] = ((orig_raw['Arrival Delay in Minutes'] + 1.0) / (orig_raw['Departure Delay in Minutes'] + 1.0)).astype(np.float32)
    orig_raw['Zero_Ratings_Count'] = (orig_raw[RATINGS] == 0).sum(axis=1).astype(np.float32)
    orig_base_cols = CORE_FEATS + ['Delay_Total', 'Delay_Diff', 'Delay_Ratio', 'Zero_Ratings_Count']
    X_orig = orig_raw[orig_base_cols].copy()
    X_eval = full_df[orig_base_cols].copy()
    for c in CATS:
        X_orig[c] = X_orig[c].astype('category')
        X_eval[c] = X_eval[c].astype('category')

    og_xgb = xgb.XGBClassifier(
        n_estimators=300 if GPU else 150,
        max_depth=8,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.60,
        enable_categorical=True,
        tree_method='hist',
        device=DEVICE,
        random_state=SEED
    )
    og_xgb.fit(X_orig, y_orig)
    og_xgb_preds = og_xgb.predict_proba(X_eval)[:, 1].astype(np.float32)

    og_lgb = lgb.LGBMClassifier(
        n_estimators=350 if GPU else 150,
        learning_rate=0.035,
        num_leaves=180,
        max_depth=9,
        subsample=0.85,
        colsample_bytree=0.65,
        n_jobs=CPU_THREADS,
        verbose=-1,
        random_state=SEED
    )
    og_lgb.fit(X_orig, y_orig)
    og_lgb_preds = og_lgb.predict_proba(X_eval)[:, 1].astype(np.float32)

    def to_logit(p, eps=1e-6):
        p = np.clip(p, eps, 1.0 - eps)
        return np.log(p / (1.0 - p))

    ORIG_PRIOR_FEATS['og_xgb_prob'] = og_xgb_preds
    ORIG_PRIOR_FEATS['og_xgb_logit'] = to_logit(og_xgb_preds).astype(np.float32)
    ORIG_PRIOR_FEATS['og_lgb_prob'] = og_lgb_preds
    ORIG_PRIOR_FEATS['og_lgb_logit'] = to_logit(og_lgb_preds).astype(np.float32)
else:
    # Fallback zero-filled priors if external 129k dataset not provided
    ORIG_PRIOR_FEATS['orig_route_satisfaction_prior'] = 0.5
    ORIG_PRIOR_FEATS['og_xgb_prob'] = 0.5
    ORIG_PRIOR_FEATS['og_xgb_logit'] = 0.0
    ORIG_PRIOR_FEATS['og_lgb_prob'] = 0.5
    ORIG_PRIOR_FEATS['og_lgb_logit'] = 0.0

core_encoded = full_df[CORE_FEATS].copy()
for c in CATS:
    core_encoded[c] = core_encoded[c].astype('category')

context_cols = [
    'Delay_Total', 'Delay_Diff', 'Delay_Ratio', 'Delay_Per_Mile',
    'Zero_Ratings_Count', 'Digital_Service_Index', 'Digital_Satisfaction_Gap',
    'Physical_Service_Index', 'Cabin_Comfort_Cluster', 'Service_Expectation_Gap',
    'Wifi_Boarding_Ratio', 'Loyal_Business_Segment'
]

STATIC_X = pd.concat([
    core_encoded,
    full_df[context_cols],
    CNT_FEATS,
    ROUTE_PROFILES,
    ORIG_PRIOR_FEATS
], axis=1)

print(f'[OK] Master Static Features Assembled: {STATIC_X.shape[1]} features')

## 4. Zero-Leakage In-Fold Bayesian Target Encodings (10 Compound Keys)
Calculated strictly within training splits of each CV fold using 5-fold inner cross-validation and empirical Bayes smoothing ($\lambda=25.0$).

In [ ]:
def combine_keys(df, cols):
    code = np.zeros(len(df), dtype=np.int64)
    for c in cols:
        k = pd.factorize(df[c], use_na_sentinel=False)[0]
        code = code * (k.max() + 1) + k
    return pd.factorize(code)[0]

TE_KEYS_CONFIG = [
    [FD],
    ['Age'],
    [FD, 'Age'],
    [FD, 'Class'],
    [FD, 'Type of Travel'],
    ['Class', 'Type of Travel', 'Inflight wifi service'],
    ['Age', 'Class', 'Type of Travel', 'Customer Type'],
    [FD, 'Class', 'Type of Travel'],             # Micro-persona per route
    [FD, 'Inflight wifi service'],               # Route connectivity delivery
    ['Class', 'Seat comfort', 'Cleanliness']     # Cabin physical comfort standard
]

TE_KEYS_DF = pd.DataFrame(index=full_df.index)
for cols in TE_KEYS_CONFIG:
    col_name = 'te_key_' + '_x_'.join(cols)
    TE_KEYS_DF[col_name] = combine_keys(full_df, cols)

def apply_bayesian_te(trn_keys, y_tr, eval_keys_list, weight=25.0):
    global_prior = float(y_tr.mean())
    fit_encodings = {}
    eval_encodings_list = [{} for _ in eval_keys_list]
    inner_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
    
    for col in trn_keys.columns:
        k_fit = trn_keys[col].values
        oof_te = np.zeros(len(trn_keys), dtype=np.float32)
        
        for in_tr, in_va in inner_cv.split(k_fit):
            stats = pd.DataFrame({'k': k_fit[in_tr], 'y': y_tr[in_tr]}).groupby('k')['y'].agg(['sum', 'count'])
            smoothed = (stats['sum'] + global_prior * weight) / (stats['count'] + weight)
            val_k = pd.Series(k_fit[in_va])
            oof_te[in_va] = val_k.map(smoothed).fillna(global_prior).values.astype(np.float32)
            
        fit_encodings[f'te_{col}'] = oof_te
        
        full_stats = pd.DataFrame({'k': k_fit, 'y': y_tr}).groupby('k')['y'].agg(['sum', 'count'])
        full_smoothed = (full_stats['sum'] + global_prior * weight) / (full_stats['count'] + weight)
        for idx, eval_df in enumerate(eval_keys_list):
            eval_k = eval_df[col]
            eval_encodings_list[idx][f'te_{col}'] = eval_k.map(full_smoothed).fillna(global_prior).values.astype(np.float32)
            
    return pd.DataFrame(fit_encodings), [pd.DataFrame(e) for e in eval_encodings_list]

# Generate 10 Stratified CV Fold Matrices
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train_raw, y_train))
X_tr_static = STATIC_X.iloc[:NTR].copy()
X_te_static = STATIC_X.iloc[NTR:].copy()
TE_tr_keys = TE_KEYS_DF.iloc[:NTR].copy()
TE_te_keys = TE_KEYS_DF.iloc[NTR:].copy()

FOLD_MATRICES = []
t_te_start = time.time()
print(f'[*] Precomputing Zero-Leakage Target Encodings across {N_FOLDS} Folds...')

for fold, (trn_idx, val_idx) in enumerate(FOLDS):
    y_tr_f = y_train[trn_idx]
    y_va_f = y_train[val_idx]
    trn_te_df, (val_te_df, te_te_df) = apply_bayesian_te(
        trn_keys=TE_tr_keys.iloc[trn_idx],
        y_tr=y_tr_f,
        eval_keys_list=[TE_tr_keys.iloc[val_idx], TE_te_keys],
        weight=25.0
    )
    X_tr_f = pd.concat([X_tr_static.iloc[trn_idx].reset_index(drop=True), trn_te_df.reset_index(drop=True)], axis=1)
    X_va_f = pd.concat([X_tr_static.iloc[val_idx].reset_index(drop=True), val_te_df.reset_index(drop=True)], axis=1)
    X_te_f = pd.concat([X_te_static.reset_index(drop=True), te_te_df.reset_index(drop=True)], axis=1)
    FOLD_MATRICES.append((X_tr_f, X_va_f, X_te_f, y_tr_f, y_va_f))

print(f'[OK] Total features per fold matrix: {FOLD_MATRICES[0][0].shape[1]} in {time.time() - t_te_start:.1f}s.')

## 5. Model Training Suite (High-Resolution Histogram Trees, max_bin=512)

In [ ]:
OOF_PREDS = {}
TEST_PREDS = {}

def train_cv_model(model_name, fit_fn):
    print(f"\n{'='*20} Training: {model_name} ({N_FOLDS} Folds) {'='*20}")
    oof = np.zeros(NTR, dtype=np.float32)
    test_accum = np.zeros(NTE, dtype=np.float32)
    t0 = time.time()
    
    for fold, (trn_idx, val_idx) in enumerate(FOLDS):
        X_tr, X_va, X_te, y_tr, y_va = FOLD_MATRICES[fold]
        predict_fn = fit_fn(X_tr, y_tr, X_va, y_va, fold)
        oof[val_idx] = predict_fn(X_va)
        test_accum += predict_fn(X_te) / len(FOLDS)
        fold_auc = roc_auc_score(y_va, oof[val_idx])
        print(f" Fold {fold + 1:2d}/{N_FOLDS} AUC: {fold_auc:.6f}")
        
    overall_auc = roc_auc_score(y_train, oof)
    print(f"[AUC] {model_name} Overall OOF AUC: {overall_auc:.6f} ({time.time() - t0:.1f}s)")
    OOF_PREDS[model_name] = oof
    TEST_PREDS[model_name] = test_accum
    gc.collect()
    if GPU:
        torch.cuda.empty_cache()

# 1. XGBoost Depth 8 Wide (Primary Champion)
def fit_xgb_d8_wide(X_tr, y_tr, X_va, y_va, fold):
    mcw = 30 if len(X_tr) > 2000 else 1
    model = xgb.XGBClassifier(
        n_estimators=N_EST,
        learning_rate=0.02,
        max_depth=8,
        subsample=0.85,
        colsample_bytree=0.60,
        colsample_bynode=0.65,
        min_child_weight=mcw,
        reg_lambda=2.5,
        max_bin=512,
        tree_method='hist',
        device=DEVICE,
        eval_metric='auc',
        early_stopping_rounds=ES,
        enable_categorical=True,
        n_jobs=CPU_THREADS,
        random_state=SEED + fold + 30
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return lambda X: model.predict_proba(X)[:, 1].astype(np.float32)

train_cv_model('XGBoost_Depth8_Wide', fit_xgb_d8_wide)

# 2. XGBoost Tuned D9 (Node-Subsampled)
def fit_xgb_tuned(X_tr, y_tr, X_va, y_va, fold):
    mcw = 55 if len(X_tr) > 2000 else 1
    model = xgb.XGBClassifier(
        n_estimators=N_EST,
        learning_rate=0.02,
        max_depth=9,
        subsample=0.80,
        colsample_bytree=0.70,
        colsample_bynode=0.50,
        min_child_weight=mcw,
        reg_lambda=3.55,
        max_bin=512,
        tree_method='hist',
        device=DEVICE,
        eval_metric='auc',
        early_stopping_rounds=ES,
        enable_categorical=True,
        n_jobs=CPU_THREADS,
        random_state=SEED + fold + 20
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return lambda X: model.predict_proba(X)[:, 1].astype(np.float32)

train_cv_model('XGBoost_Tuned_D9', fit_xgb_tuned)

# 3. XGBoost Depth 10 Anchor
def fit_xgb_d10(X_tr, y_tr, X_va, y_va, fold):
    mcw = 20 if len(X_tr) > 2000 else 1
    model = xgb.XGBClassifier(
        n_estimators=N_EST,
        learning_rate=0.02,
        max_depth=10,
        subsample=0.85,
        colsample_bytree=0.80,
        min_child_weight=mcw,
        reg_lambda=3.0,
        tree_method='hist',
        device=DEVICE,
        eval_metric='auc',
        early_stopping_rounds=ES,
        enable_categorical=True,
        n_jobs=CPU_THREADS,
        random_state=SEED + fold + 10
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return lambda X: model.predict_proba(X)[:, 1].astype(np.float32)

train_cv_model('XGBoost_Depth10_Anchor', fit_xgb_d10)

# 4. LightGBM Deep (255 Leaves)
def fit_lgb_deep(X_tr, y_tr, X_va, y_va, fold):
    mcs = 60 if len(X_tr) > 2000 else 5
    model = lgb.LGBMClassifier(
        n_estimators=N_EST,
        learning_rate=0.02,
        num_leaves=255,
        max_depth=10,
        min_child_samples=mcs,
        subsample=0.80,
        subsample_freq=1,
        colsample_bytree=0.75,
        reg_lambda=4.0,
        max_bin=255,
        n_jobs=CPU_THREADS,
        verbose=-1,
        random_state=SEED + fold + 40
    )
    model.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc',
        callbacks=[lgb.early_stopping(ES, verbose=False)]
    )
    return lambda X: model.predict_proba(X)[:, 1].astype(np.float32)

train_cv_model('LightGBM_Deep', fit_lgb_deep)

## 6. Multi-Champion Ensembling (Caruana Selection & Consensus Blending)

In [ ]:
# Strategy 1: Caruana Forward ROC AUC Selection (50 iterations)
def caruana_forward_auc(y_true, pred_dict, max_iters=50):
    names = list(pred_dict.keys())
    weights = {m: 0 for m in names}
    current_blend = np.zeros(len(y_true), dtype=np.float64)
    best_auc = 0.0
    for it in range(max_iters):
        best_cand = None
        best_cand_auc = -1.0
        for m in names:
            cand_blend = (current_blend * it + pred_dict[m]) / (it + 1)
            auc = roc_auc_score(y_true, cand_blend)
            if auc > best_cand_auc:
                best_cand_auc = auc
                best_cand = m
        current_blend = (current_blend * it + pred_dict[best_cand]) / (it + 1)
        weights[best_cand] += 1
        best_auc = best_cand_auc
    normalized_weights = {m: weights[m] / max_iters for m in names}
    return normalized_weights, current_blend, best_auc

caruana_weights, oof_caruana, auc_caruana = caruana_forward_auc(y_train, OOF_PREDS, max_iters=50)
te_caruana = np.zeros(NTE, dtype=np.float64)
for m, w in caruana_weights.items():
    if w > 0:
        te_caruana += w * TEST_PREDS[m]

# Strategy 2: Pure XGBoost Trio (45% Wide + 45% D9 + 10% Depth 10)
oof_trio = 0.45 * OOF_PREDS['XGBoost_Depth8_Wide'] + 0.45 * OOF_PREDS['XGBoost_Tuned_D9'] + 0.10 * OOF_PREDS['XGBoost_Depth10_Anchor']
te_trio  = 0.45 * TEST_PREDS['XGBoost_Depth8_Wide'] + 0.45 * TEST_PREDS['XGBoost_Tuned_D9'] + 0.10 * TEST_PREDS['XGBoost_Depth10_Anchor']
auc_trio = roc_auc_score(y_train, oof_trio)

# Strategy 3: Pure Regularized XGBoost Duo (50% Wide + 50% D9)
oof_duo = 0.50 * OOF_PREDS['XGBoost_Depth8_Wide'] + 0.50 * OOF_PREDS['XGBoost_Tuned_D9']
te_duo  = 0.50 * TEST_PREDS['XGBoost_Depth8_Wide'] + 0.50 * TEST_PREDS['XGBoost_Tuned_D9']
auc_duo = roc_auc_score(y_train, oof_duo)

# Strategy 4: 50/50 Consensus Blend (Caruana Champion + Pure XGBoost Trio)
oof_consensus = 0.50 * oof_caruana + 0.50 * oof_trio
te_consensus  = 0.50 * te_caruana + 0.50 * te_trio
auc_consensus = roc_auc_score(y_train, oof_consensus)

print('\n' + '='*50)
print('ENSEMBLE PERFORMANCE COMPARISON (ROC-AUC):')
print(f' * Pure Regularized XGB Duo (50/50): {auc_duo:.6f}')
print(f' * Pure XGBoost Trio (45/45/10):    {auc_trio:.6f}')
print(f' * Caruana Forward Selection:       {auc_caruana:.6f}')
print(f' * 50/50 Consensus Blend:           {auc_consensus:.6f}')
print('='*50)
print('\nCaruana Weight Allocations:')
for m, w in caruana_weights.items():
    print(f' * {m:26s}: {w*100:5.1f}%')

## 7. Generate All 5 Verified Submission Files

In [ ]:
os.makedirs('submissions', exist_ok=True)
test_ids = test_raw['id'].values if 'id' in test_raw.columns else test_raw.index

# 1. Primary Consensus Champion
sub = pd.DataFrame({'id': test_ids, TARGET: te_consensus})
sub.to_csv('submissions/submission.csv', index=False)

# 2. Caruana Standalone Champion
sub_caruana = pd.DataFrame({'id': test_ids, TARGET: te_caruana})
sub_caruana.to_csv('submissions/submission_caruana.csv', index=False)

# 3. Pure XGBoost Trio
sub_trio = pd.DataFrame({'id': test_ids, TARGET: te_trio})
sub_trio.to_csv('submissions/submission_xgb_trio.csv', index=False)

# 4. Pure XGBoost Duo
sub_duo = pd.DataFrame({'id': test_ids, TARGET: te_duo})
sub_duo.to_csv('submissions/submission_duo_xgb.csv', index=False)

# 5. Percentile Rank-Normalized Variant
sub_rank = pd.DataFrame({'id': test_ids, TARGET: rankdata(te_consensus).astype(np.float32) / len(test_raw)})
sub_rank.to_csv('submissions/submission_rank.csv', index=False)

# Integrity Verification
for fname, df_sub in [
    ('submission.csv', sub),
    ('submission_caruana.csv', sub_caruana),
    ('submission_xgb_trio.csv', sub_trio),
    ('submission_duo_xgb.csv', sub_duo),
    ('submission_rank.csv', sub_rank)
]:
    assert len(df_sub) == len(test_raw), f'Length mismatch in {fname}'
    assert df_sub[TARGET].isna().sum() == 0, f'Nulls found in {fname}'
    assert (df_sub[TARGET] >= 0.0).all() and (df_sub[TARGET] <= 1.0).all(), f'Out of bounds in {fname}'

print('[*] All 5 submissions generated and verified under submissions/:')
print(' * submissions/submission.csv')
print(' * submissions/submission_caruana.csv')
print(' * submissions/submission_xgb_trio.csv')
print(' * submissions/submission_duo_xgb.csv')
print(' * submissions/submission_rank.csv')
print('\nPrimary Submission Head:')
display(sub.head(10))

In [ ]:
# ==============================================================================
# AUTOMATIC DOWNLOADING & GOOGLE DRIVE BACKUP ON COMPLETION
# ==============================================================================
import os, glob, shutil
print('[*] Pipeline complete! Preparing automatic download and Drive backup...')

# 1. If Google Drive is mounted, copy all submissions and OOF arrays to Drive
drive_dest = '/content/drive/MyDrive/kaggle_airline_output'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_dest, exist_ok=True)
    for f in glob.glob('submissions/*.csv') + glob.glob('oof/*.npy'):
        shutil.copy(f, drive_dest)
    print(f'[OK] Backed up all outputs to Google Drive: {drive_dest}')

# 2. Automatically trigger browser downloads for each generated submission CSV
try:
    from google.colab import files
    sub_files = glob.glob('submissions/*.csv')
    print(f'[*] Triggering browser download for {len(sub_files)} submission files:')
    for f in sorted(sub_files):
        print(f'  - Downloading: {f}')
        files.download(f)
    print('[OK] Download requests dispatched to browser.')
except ImportError:
    print('[*] Local environment detected; submissions saved in submissions/ directory.')
